In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yutkin/corpus-of-russian-news-articles-from-lenta")

print("Path to dataset files:", path)

100%|██████████| 584M/584M [00:08<00:00, 75.9MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/yutkin/corpus-of-russian-news-articles-from-lenta/versions/2


In [2]:
# @title Установка библиотек
!pip install -q corus

import os
import re
import warnings
import numpy as np
import pandas as pd
import corus

from nltk.stem.snowball import SnowballStemmer
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Фиксация сидов для полной воспроизводимости
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
np.random.seed(SEED)
warnings.filterwarnings('ignore')

print("✅ Библиотеки загружены, seed зафиксирован.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 4.3 MB/s eta 0:00:00
✅ Библиотеки загружены, seed зафиксирован.


In [3]:
!pip install -q datasets

In [4]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Путь, полученный от kagglehub
dataset_path = "/root/.cache/kagglehub/datasets/yutkin/corpus-of-russian-news-articles-from-lenta/versions/2"

print(f"🔍 Поиск файлов в: {dataset_path}")
files = os.listdir(dataset_path)
print(f"📂 Найденные файлы: {files}")

# Ищем CSV файл
csv_file = None
for f in files:
    if f.endswith('.csv'):
        csv_file = os.path.join(dataset_path, f)
        break

if not csv_file:
    raise FileNotFoundError("CSV файл не найден в скачанной директории.")

print(f"✅ Загрузка файла: {csv_file}")
df = pd.read_csv(csv_file)

# --- ПРОВЕРКА И ОЧИСТКА ДАННЫХ ---
print(f"📊 Исходные колонки: {df.columns.tolist()}")

# Приводим названия колонок к единому виду
col_map = {}
for c in df.columns:
    cl = c.lower().strip()
    if 'title' in cl or 'header' in cl:
        col_map['title'] = c
    elif 'text' in cl or 'body' in cl or 'content' in cl:
        col_map['text'] = c
    elif 'topic' in cl or 'rubric' in cl or 'category' in cl:
        col_map['topic'] = c

if len(col_map) < 3:
    print(f"Найденные колонки: {df.columns.tolist()}")
    raise ValueError(f"Не удалось определить колонки title, text, topic. Найдено: {col_map}")

df.rename(columns=col_map, inplace=True)
df = df[['title', 'text', 'topic']].dropna()

# Очистка топиков
df['topic'] = df['topic'].astype(str).str.strip()
df = df[df['topic'].isin(['', 'nan', 'None']) == False]

# Объединяем текст для лучшего контекста
df['full_text'] = df['title'] + ' ' + df['text']

print(f"📊 Размер до фильтрации: {len(df):,} записей")
print(f"🏷️ Уникальных топиков до фильтрации: {df['topic'].nunique()}")

#  ФИЛЬТРАЦИЯ РЕДКИХ КЛАССОВ ПЕРЕД СЭМПЛИРОВАНИЕМ ---
min_class_size = 50
initial_count = len(df)
# Оставляем только топики, где >= 50 новостей
df = df.groupby('topic').filter(lambda x: len(x) >= min_class_size)
df = df.reset_index(drop=True)

removed_count = initial_count - len(df)
print(f"📉 Размер после фильтрации редких классов (<{min_class_size}): {len(df):,}")
if removed_count > 0:
    print(f"   (Удалено {removed_count} записей из редких классов)")

print(f"🏷️ Уникальных топиков после фильтрации: {df['topic'].nunique()}")
print(f"📈 Топ-5 топиков:\n{df['topic'].value_counts().head()}")

# --- СЭМПЛИРОВАНИЕ ---
SAMPLE_SIZE = 100_000
SEED = 42

if len(df) > SAMPLE_SIZE:
    print(f"⏳ Стратифицированное сэмплирование до {SAMPLE_SIZE} записей...")
    # Теперь stratify безопасен, так как все классы имеют >= 50 примеров
    df_sampled, _ = train_test_split(
        df,
        train_size=SAMPLE_SIZE,
        stratify=df['topic'],
        random_state=SEED
    )
else:
    print("Датасет меньше 100k, берем целиком.")
    df_sampled = df.copy()

# Финальная проверка перед сплитом на train/val/test
print(f"✅ Данные готовы к предобработке. Размер выборки: {len(df_sampled)}")

🔍 Поиск файлов в: /root/.cache/kagglehub/datasets/yutkin/corpus-of-russian-news-articles-from-lenta/versions/2
📂 Найденные файлы: ['lenta-ru-news.csv']
✅ Загрузка файла: /root/.cache/kagglehub/datasets/yutkin/corpus-of-russian-news-articles-from-lenta/versions/2/lenta-ru-news.csv
📊 Исходные колонки: ['url', 'title', 'text', 'topic', 'tags', 'date']
📊 Размер до фильтрации: 738,968 записей
🏷️ Уникальных топиков до фильтрации: 23
📉 Размер после фильтрации редких классов (<50): 738,961
   (Удалено 7 записей из редких классов)
🏷️ Уникальных топиков после фильтрации: 19
📈 Топ-5 топиков:
topic
Россия       160442
Мир          136620
Экономика     79528
Спорт         64413
Культура      53796
Name: count, dtype: int64
⏳ Стратифицированное сэмплирование до 100000 записей...
✅ Данные готовы к предобработке. Размер выборки: 100000


In [5]:
# Компилируем регексы для ускорения
RE_NOISE = re.compile(r'http\S+|www\S+|https\S+|<.*?>|\d+|[^\w\s]')
RE_SPACE = re.compile(r'\s+')
stemmer = SnowballStemmer("russian")

def preprocess_text(text: str) -> str:
    text = text.lower()
    text = RE_NOISE.sub(' ', text)
    text = RE_SPACE.sub(' ', text).strip()
    # Стемминг + фильтр коротких слов
    return ' '.join(stemmer.stem(w) for w in text.split() if len(w) > 2)

print("⏳ Запуск предобработки (1-2 мин для 100k строк)...")
df_sampled['clean_text'] = df_sampled['full_text'].apply(preprocess_text)

# Кодирование таргета
le = LabelEncoder()
df_sampled['target'] = le.fit_transform(df_sampled['topic'])
print(f"✅ Таргет закодирован. Классы: {le.classes_}")

# Разделение 60 / 20 / 20 со стратификацией
X = df_sampled['clean_text']
y = df_sampled['target']

X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, stratify=y_train_val, random_state=SEED
)  # 0.25 от 0.8 = 0.2

print(f"📐 Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")

⏳ Запуск предобработки (1-2 мин для 100k строк)...
✅ Таргет закодирован. Классы: ['69-я параллель' 'Библиотека' 'Бизнес' 'Бывший СССР' 'Дом' 'Из жизни'
 'Интернет и СМИ' 'Крым' 'Культпросвет' 'Культура' 'Легпром' 'Мир'
 'Наука и техника' 'Путешествия' 'Россия' 'Силовые структуры' 'Спорт'
 'Ценности' 'Экономика']
📐 Train: 60,000 | Val: 20,000 | Test: 20,000


In [6]:
dummy = DummyClassifier(strategy='most_frequent', random_state=SEED)
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_val)

print("🟤 Dummy Baseline (most_frequent):")
print(f"   Accuracy : {accuracy_score(y_val, y_pred_dummy):.4f}")
print(f"   Macro F1 : {f1_score(y_val, y_pred_dummy, average='macro'):.4f}")

🟤 Dummy Baseline (most_frequent):
   Accuracy : 0.2172
   Macro F1 : 0.0188


In [7]:
# Базовые пайплайны
pipe_count = Pipeline([
    ('vec', CountVectorizer(max_features=50000, min_df=5, max_df=0.95, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=SEED))
])

pipe_tfidf = Pipeline([
    ('vec', TfidfVectorizer(max_features=50000, min_df=5, max_df=0.95, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf', LogisticRegression(max_iter=1000, solver='saga', n_jobs=-1, random_state=SEED))
])

print("⏳ Обучение CountVectorizer + LogReg...")
pipe_count.fit(X_train, y_train)
pred_count = pipe_count.predict(X_val)
print(f"✅ CountVec  | Acc: {accuracy_score(y_val, pred_count):.4f} | Macro F1: {f1_score(y_val, pred_count, average='macro'):.4f}")

print("⏳ Обучение TfidfVectorizer + LogReg...")
pipe_tfidf.fit(X_train, y_train)
pred_tfidf = pipe_tfidf.predict(X_val)
print(f"✅ TfidfVec  | Acc: {accuracy_score(y_val, pred_tfidf):.4f} | Macro F1: {f1_score(y_val, pred_tfidf, average='macro'):.4f}")

⏳ Обучение CountVectorizer + LogReg...
✅ CountVec  | Acc: 0.8165 | Macro F1: 0.6237
⏳ Обучение TfidfVectorizer + LogReg...
✅ TfidfVec  | Acc: 0.8161 | Macro F1: 0.5437


In [8]:
# TF-IDF показал себя лучше, тюним его
param_grid = {
    'vec__max_features': [30000, 50000],
    'vec__ngram_range': [(1, 1), (1, 2)],
    'vec__min_df': [3, 5],
    'clf__C': [0.5, 1.0, 2.0],
    'clf__penalty': ['l2']  # saga поддерживает l2, l1, elasticnet
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
grid = GridSearchCV(
    pipe_tfidf, param_grid, cv=cv, scoring='f1_macro',
    n_jobs=-1, verbose=1, refit=True
)

print("⏳ Запуск GridSearchCV (3-fold)...")
grid.fit(X_train, y_train)

print(f"\n🏆 Best Params: {grid.best_params_}")
print(f"📈 Best CV Macro F1: {grid.best_score_:.4f}")

⏳ Запуск GridSearchCV (3-fold)...
Fitting 3 folds for each of 24 candidates, totalling 72 fits

🏆 Best Params: {'clf__C': 2.0, 'clf__penalty': 'l2', 'vec__max_features': 50000, 'vec__min_df': 5, 'vec__ngram_range': (1, 1)}
📈 Best CV Macro F1: 0.5541


In [9]:
best_model = grid.best_estimator_
y_pred_test = best_model.predict(X_test)

test_acc = accuracy_score(y_test, y_pred_test)
test_f1 = f1_score(y_test, y_pred_test, average='macro')

print("📊 Итоговое качество на TEST выборке:")
print(f"   Accuracy : {test_acc:.4f}")
print(f"   Macro F1 : {test_f1:.4f}\n")

print("📋 Classification Report (топ-10 классов по support):")
report = classification_report(y_test, y_pred_test, target_names=le.classes_, output_dict=True)
df_report = pd.DataFrame(report).transpose()
print(df_report.sort_values('support', ascending=False).head(10).to_string())

📊 Итоговое качество на TEST выборке:
   Accuracy : 0.8176
   Macro F1 : 0.5783

📋 Classification Report (топ-10 классов по support):
                 precision    recall  f1-score  support
macro avg         0.699240  0.546208  0.578304  20000.0
weighted avg      0.815996  0.817650  0.812364  20000.0
Россия            0.773637  0.846154  0.808272   4342.0
Мир               0.794548  0.851271  0.821932   3698.0
Экономика         0.834529  0.864777  0.849384   2152.0
Спорт             0.962243  0.964450  0.963345   1744.0
Культура          0.858674  0.880495  0.869447   1456.0
Бывший СССР       0.830061  0.848443  0.839151   1445.0
Наука и техника   0.841851  0.847705  0.844768   1438.0
Интернет и СМИ    0.779124  0.691481  0.732691   1209.0
